In [1]:
import os
import cv2
import numpy as np
from tqdm import tqdm
from PIL import Image
import albumentations as A


class IndustrialCompostAugmenter:
    def __init__(self, input_dirs, output_base_dir):
        self.input_dirs = [d.rstrip("/") for d in input_dirs]
        self.output_base_dir = output_base_dir.rstrip("/")
        
        self.transform_pipeline = A.Compose([
            A.Rotate(limit=90, p=0.5),
            A.HorizontalFlip(p=0.3),
            A.VerticalFlip(p=0.3),
            A.Affine(
                translate_percent=0.15,
                scale=0.15,
                rotate=30,
                shear=10,
                p=0.4,
            ),
            A.OpticalDistortion(distort_limit=0.3, p=0.3),
            A.CoarseDropout(
                num_holes_range=(1, 5),
                hole_height_range=(1, 24),
                hole_width_range=(1, 24),
                fill=0,
                p=0.3,
            ),
            A.RandomBrightnessContrast(
                brightness_limit=0.08,
                contrast_limit=0.08,
                p=0.4,
            ),
        ])

    def read_image(self, img_path):
        try:
            pil_img = Image.open(img_path)
            if pil_img.mode != "RGB":
                pil_img = pil_img.convert("RGB")
            return cv2.cvtColor(np.array(pil_img), cv2.COLOR_RGB2BGR)
        except Exception as e:
            print(f"Failed to read image {img_path}: {str(e)}")
            return None

    def process_image(self, img_path, output_dir, image_id, variants=20):
        try:
            image = self.read_image(img_path)
            if image is None:
                return 0
            
            image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
            
            generated_count = 0
            for i in range(variants):
                augmented = self.transform_pipeline(image=image_rgb)["image"]
                
                base_name = os.path.basename(img_path)
                name_without_ext = os.path.splitext(base_name)[0]
                output_path = os.path.join(
                    output_dir,
                    f"{name_without_ext}_{image_id:04d}_{i:02d}.jpg"
                )
                
                cv2.imwrite(
                    output_path,
                    cv2.cvtColor(augmented, cv2.COLOR_RGB2BGR),
                    [cv2.IMWRITE_JPEG_QUALITY, 95]
                )
                generated_count += 1
                
            return generated_count
            
        except Exception as e:
            print(f"Augmentation failed {img_path}: {str(e)}")
            return 0

    def augment_images(self, target_per_class=500, variants_per_image=20):
        os.makedirs(self.output_base_dir, exist_ok=True)
        
        total_generated = 0
        total_original = 0
        
        for input_dir in self.input_dirs:
            if not os.path.exists(input_dir):
                print(f"Warning: Directory does not exist {input_dir}")
                continue
            
            class_name = os.path.basename(input_dir)
            class_output_dir = os.path.join(self.output_base_dir, class_name)
            
            os.makedirs(class_output_dir, exist_ok=True)
            
            class_images = []
            for root, _, files in os.walk(input_dir):
                for file in files:
                    if file.lower().endswith(('.jpg', '.jpeg', '.png', '.bmp')):
                        img_path = os.path.join(root, file)
                        class_images.append(img_path)
            
            if not class_images:
                print(f"Warning: No image files found in directory {input_dir}")
                continue
            
            print(f"\nProcessing class: {class_name}")
            print(f"Number of original images: {len(class_images)}")
            
            if len(class_images) * variants_per_image < target_per_class:
                adjusted_variants = (target_per_class + len(class_images) - 1) // len(class_images)
                print(f"Adjusted variants per image: {variants_per_image} -> {adjusted_variants}")
                current_variants = adjusted_variants
            else:
                current_variants = variants_per_image
            
            class_generated = 0
            progress_bar = tqdm(total=len(class_images), desc=f"Augmenting {class_name}")
            
            for idx, img_path in enumerate(class_images):
                generated = self.process_image(
                    img_path,
                    class_output_dir,
                    idx,
                    current_variants
                )
                class_generated += generated
                progress_bar.update(1)
            
            progress_bar.close()
            
            total_original += len(class_images)
            total_generated += class_generated
            
            print(f"Class {class_name} completed:")
            print(f"  Original images: {len(class_images)}")
            print(f"  Generated images: {class_generated}")
            print(f"  Output directory: {class_output_dir}")
        
        print(f"\nAll classes augmentation completed!")
        print(f"Total original images: {total_original}")
        print(f"Total generated images: {total_generated}")
        print(f"Output base directory: {self.output_base_dir}")
        
        print(f"\nOutput directory structure:")
        for class_name in os.listdir(self.output_base_dir):
            class_dir = os.path.join(self.output_base_dir, class_name)
            if os.path.isdir(class_dir):
                file_count = len([f for f in os.listdir(class_dir) 
                                if f.lower().endswith(('.jpg', '.jpeg', '.png', '.bmp'))])
                print(f"  {class_name}/ - {file_count} images")


if __name__ == "__main__":
    INPUT_DIRS = [
        r"E:\TSG\jupyterlab\machine learning image\Classification\immature",
        r"E:\TSG\jupyterlab\machine learning image\Classification\mature",
    ]
    
    OUTPUT_BASE_DIR = r"E:\TSG\jupyterlab\machine learning image\all"
    
    augmenter = IndustrialCompostAugmenter(INPUT_DIRS, OUTPUT_BASE_DIR)
    
    augmenter.augment_images(
        target_per_class=500,
        variants_per_image=20
    )


Processing class: immature
Number of original images: 143


Augmenting immature: 100%|███████████████████████████████████████████████████████████| 143/143 [01:53<00:00,  1.26it/s]


Class immature completed:
  Original images: 143
  Generated images: 2860
  Output directory: E:\TSG\jupyterlab\machine learning image\all\immature

Processing class: mature
Number of original images: 63


Augmenting mature: 100%|███████████████████████████████████████████████████████████████| 63/63 [00:47<00:00,  1.31it/s]

Class mature completed:
  Original images: 63
  Generated images: 1260
  Output directory: E:\TSG\jupyterlab\machine learning image\all\mature

All classes augmentation completed!
Total original images: 206
Total generated images: 4120
Output base directory: E:\TSG\jupyterlab\machine learning image\all

Output directory structure:
  .ipynb_checkpoints/ - 0 images
  immature/ - 120 images
  mature/ - 0 images
